# 04 — Species Head (TMA field, Real OpenFOAM Data)

Trains a MeshGraphNet to predict the **TMA mass fraction field** from the 80 reactingFoam ALD cases.

**Data:** 80 HDF5 files in Google Drive (`showerhead_openfoam/`)  
**Target:** TMA field — mass fraction of trimethylaluminium precursor  
**Architecture:** MeshGraphNet — hidden=128, 8 processor layers  
**Key physics:** Pe_m = Re·Sc governs convection vs. diffusion; Da drives reaction-limited vs. transport-limited regime  
**Requires:** Colab A100 GPU

## 0. Setup

In [ ]:
import subprocess, sys, os

subprocess.run(['pip', 'install', '-q', 'torch_geometric', 'h5py', 'scipy', 'tqdm'], check=True)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/Ranaam21/cfd-ald-app.git'
REPO_DIR = '/content/cfd-ald-app'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or 'Repo up to date.')
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Setup done.')

## 1. Config

In [ ]:
import torch
from pathlib import Path

assert torch.cuda.is_available(), 'Switch to A100 GPU runtime first'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── Paths ───────────────────────────────────────────────────────────────────
DRIVE_BASE = Path('/content/drive/MyDrive/cfd-ald-app')
HDF5_DIR   = DRIVE_BASE / 'showerhead_openfoam'
CKPT_DIR   = DRIVE_BASE / 'checkpoints' / 'species_head'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

h5_files = sorted(HDF5_DIR.glob('case_*.h5'))
print(f'Found {len(h5_files)} HDF5 case files in {HDF5_DIR}')
assert len(h5_files) > 0, f'No HDF5 files found in {HDF5_DIR}'

# ── Model / training config ─────────────────────────────────────────────────
CFG = {
    'node_input_dim':  22,    # 4 node features + 18 global broadcast
    'edge_input_dim':  4,     # dx, dy, dz, dist (normalised by median edge length)
    'hidden_dim':      128,
    'n_layers':        8,
    'output_dim':      1,     # TMA mass fraction

    'k_neighbors':     6,
    'train_frac':      0.875, # 70 train / 10 val

    'epochs':          80,
    'lr':              3e-4,
    'grad_accum':      4,
    'grad_clip':       1.0,
    'bf16':            True,
    'save_every':      20,
    'log_every':       5,
}
print('Config ready.')

## 2. Load HDF5 data

Target is **TMA** (column 5 in `outputs/node_fields`).  
TMA is a mass fraction in [0, ~0.01]. The wafer BC is `fixedValue 0` (full consumption),
so TMA = 0 at z ≈ 0. Skip cases where TMA never exceeds 1e-5 (purge-only snapshot).

In [ ]:
import h5py
import numpy as np
from tqdm import tqdm

OUTPUT_COLS = ['Ux', 'Uy', 'Uz', 'p', 'T', 'TMA']
TMA_IDX     = OUTPUT_COLS.index('TMA')   # column 5

def load_case(h5_path: Path) -> dict:
    with h5py.File(h5_path, 'r') as f:
        coords       = f['coords'][:]               # [N, 3]
        node_feats   = f['inputs/node_features'][:] # [N, 4]
        global_feats = f['inputs/global'][:]        # [18]
        out_fields   = f['outputs/node_fields'][:]  # [N, 6]

    N = len(coords)
    global_broadcast = np.tile(global_feats, (N, 1)).astype(np.float32)  # [N, 18]
    node_input = np.concatenate([node_feats, global_broadcast], axis=1)  # [N, 22]
    TMA = out_fields[:, TMA_IDX]   # [N]

    return {
        'coords':     coords.astype(np.float32),
        'node_input': node_input,
        'TMA':        TMA.astype(np.float32),
        'name':       h5_path.stem,
    }

print('Loading HDF5 cases...')
all_cases, failed = [], []
for p in tqdm(h5_files):
    try:
        c = load_case(p)
        # Skip purge-only snapshots (TMA never appears in domain)
        if c['TMA'].max() > 1e-5:
            all_cases.append(c)
        else:
            failed.append(f'{p.name}: TMA.max={c["TMA"].max():.2e} (purge snapshot?)')
    except Exception as e:
        failed.append(f'{p.name}: {e}')

print(f'Loaded {len(all_cases)} valid cases  ({len(failed)} skipped)')
if failed:
    for f in failed[:5]:
        print(f'  {f}')

import random
random.seed(42)
random.shuffle(all_cases)
n_train = int(len(all_cases) * CFG['train_frac'])
train_cases = all_cases[:n_train]
val_cases   = all_cases[n_train:]
print(f'Train: {len(train_cases)}   Val: {len(val_cases)}')

c = train_cases[0]
print(f"\nSample case: {c['name']}")
print(f"  N cells:    {len(c['coords']):,}")
print(f"  TMA range:  {c['TMA'].min():.4e} – {c['TMA'].max():.4e}")
print(f"  TMA std:    {c['TMA'].std():.4e}")

## 3. Normalisation stats

TMA is O(0.01) — normalise so the network sees values near zero with unit variance.

In [ ]:
norm_path = CKPT_DIR / 'species_norm_stats.npz'

if norm_path.exists():
    stats     = np.load(norm_path)
    node_mean = stats['node_mean']; node_std = stats['node_std']
    TMA_mean  = float(stats['TMA_mean']); TMA_std = float(stats['TMA_std'])
    print('Loaded norm stats from checkpoint.')
else:
    sample    = train_cases[:20]
    all_nodes = np.concatenate([c['node_input'] for c in sample], axis=0)
    all_TMA   = np.concatenate([c['TMA']        for c in sample], axis=0)

    node_mean = all_nodes.mean(0).astype(np.float32)
    node_std  = np.clip(all_nodes.std(0), 1e-6, None).astype(np.float32)
    TMA_mean  = float(all_TMA.mean())
    TMA_std   = float(max(all_TMA.std(), 1e-8))

    np.savez(norm_path, node_mean=node_mean, node_std=node_std,
             TMA_mean=TMA_mean, TMA_std=TMA_std)
    print('Computed and saved norm stats.')

print(f'TMA_mean={TMA_mean:.4e}   TMA_std={TMA_std:.4e}')

## 4. Graph builder

In [ ]:
from scipy.spatial import cKDTree

K = CFG['k_neighbors']

def make_graph(case: dict) -> dict:
    coords     = case['coords']     # [N, 3]
    node_input = case['node_input'] # [N, 22]
    TMA        = case['TMA']        # [N]
    N          = len(coords)

    tree = cKDTree(coords)
    _, idx = tree.query(coords, k=K + 1)
    idx = idx[:, 1:]   # exclude self

    src = np.repeat(np.arange(N), K)
    dst = idx.flatten()

    diff = coords[dst] - coords[src]                          # [E, 3]
    dist = np.linalg.norm(diff, axis=1, keepdims=True)        # [E, 1]
    med  = float(np.median(dist)) + 1e-8
    edge_feats = np.concatenate([diff / med, dist / med], axis=1).astype(np.float32)  # [E, 4]

    node_norm = ((node_input - node_mean) / node_std).astype(np.float32)
    TMA_norm  = ((TMA - TMA_mean) / TMA_std).astype(np.float32)

    return {
        'x':          torch.from_numpy(node_norm),
        'edge_index': torch.tensor(np.stack([src, dst]), dtype=torch.long),
        'edge_attr':  torch.from_numpy(edge_feats),
        'y':          torch.from_numpy(TMA_norm).unsqueeze(1),  # [N, 1]
    }

g = make_graph(train_cases[0])
print('Graph smoke test:')
print(f'  x:          {tuple(g["x"].shape)}')
print(f'  edge_index: {tuple(g["edge_index"].shape)}')
print(f'  edge_attr:  {tuple(g["edge_attr"].shape)}')
print(f'  y:          {tuple(g["y"].shape)}')

## 5. MeshGraphNet model (SpeciesMGN)

Identical architecture to HeatMGN — same encoder/processor/decoder structure.
Separate weights because species transport (Pe_m, Da, Sc) differs from heat (Pe_h, Pr, Nu).

In [ ]:
import torch.nn as nn
from torch_geometric.nn import MessagePassing

def mlp(in_dim, out_dim, hidden=None, n_layers=2):
    hidden = hidden or out_dim
    dims = [in_dim] + [hidden] * (n_layers - 1) + [out_dim]
    mods = []
    for i in range(len(dims) - 1):
        mods.append(nn.Linear(dims[i], dims[i+1]))
        if i < len(dims) - 2:
            mods.append(nn.SiLU())
    return nn.Sequential(*mods)


class MGNProcessor(MessagePassing):
    def __init__(self, hidden):
        super().__init__(aggr='sum')
        self.edge_mlp  = mlp(3 * hidden, hidden)
        self.node_mlp  = mlp(2 * hidden, hidden)
        self.edge_norm = nn.LayerNorm(hidden)
        self.node_norm = nn.LayerNorm(hidden)

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        N     = x.size(0)
        new_e = self.edge_mlp(torch.cat([x[src], x[dst], edge_attr], dim=-1))
        new_e = self.edge_norm(new_e + edge_attr)
        agg   = self.propagate(edge_index, x=x, edge_attr=new_e, size=(N, N))
        new_x = self.node_mlp(torch.cat([x, agg], dim=-1))
        new_x = self.node_norm(new_x + x)
        return new_x, new_e

    def message(self, edge_attr):
        return edge_attr


class SpeciesMGN(nn.Module):
    def __init__(self, node_dim, edge_dim, out_dim=1, hidden=128, n_layers=8):
        super().__init__()
        self.node_enc   = mlp(node_dim, hidden)
        self.edge_enc   = mlp(edge_dim, hidden)
        self.processors = nn.ModuleList([MGNProcessor(hidden) for _ in range(n_layers)])
        self.decoder    = mlp(hidden, out_dim)

    def forward(self, x, edge_index, edge_attr):
        x = self.node_enc(x)
        e = self.edge_enc(edge_attr)
        for proc in self.processors:
            x, e = proc(x, edge_index, e)
        return self.decoder(x)


model = SpeciesMGN(
    node_dim = CFG['node_input_dim'],
    edge_dim = CFG['edge_input_dim'],
    out_dim  = CFG['output_dim'],
    hidden   = CFG['hidden_dim'],
    n_layers = CFG['n_layers'],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'SpeciesMGN: {n_params/1e6:.2f}M parameters')
print(f'hidden={CFG["hidden_dim"]}, layers={CFG["n_layers"]}')

## 6. Training

In [ ]:
import time

resume_ckpt = sorted(CKPT_DIR.glob('species_epoch*.pt'))
start_epoch = 0
train_losses, val_losses = [], []

if resume_ckpt:
    ckpt = torch.load(resume_ckpt[-1], map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    start_epoch  = ckpt['epoch'] + 1
    train_losses = ckpt.get('train_losses', [])
    val_losses   = ckpt.get('val_losses',   [])
    print(f'Resumed from epoch {start_epoch}  '
          f'(train={train_losses[-1]:.4f}, val={val_losses[-1]:.4f})')
else:
    print('Training from scratch.')

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=1e-5
)
for _ in range(start_epoch):
    scheduler.step()

dtype = torch.bfloat16 if CFG['bf16'] else torch.float32
G     = CFG['grad_accum']

print(f'\nTraining {CFG["epochs"]-start_epoch} more epochs '
      f'({len(train_cases)} train / {len(val_cases)} val cases)')
t0 = time.time()

for epoch in range(start_epoch, CFG['epochs']):
    model.train()
    random.shuffle(train_cases)
    epoch_loss = 0.0
    optimizer.zero_grad()

    for step, case in enumerate(train_cases):
        g = make_graph(case)
        x          = g['x'].to(DEVICE)
        edge_index = g['edge_index'].to(DEVICE)
        edge_attr  = g['edge_attr'].to(DEVICE)
        y          = g['y'].to(DEVICE)

        with torch.amp.autocast('cuda', dtype=dtype):
            pred = model(x, edge_index, edge_attr)
            loss = nn.functional.mse_loss(pred, y) / G

        loss.backward()

        if (step + 1) % G == 0 or step == len(train_cases) - 1:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            optimizer.step()
            optimizer.zero_grad()

        epoch_loss += loss.item() * G
        del x, edge_index, edge_attr, y, pred, loss
        torch.cuda.empty_cache()

    scheduler.step()
    train_losses.append(epoch_loss / len(train_cases))

    if (epoch + 1) % CFG['log_every'] == 0 or epoch == start_epoch:
        model.eval()
        with torch.no_grad():
            vl = 0.0
            for case in val_cases:
                g = make_graph(case)
                pred = model(g['x'].to(DEVICE), g['edge_index'].to(DEVICE),
                             g['edge_attr'].to(DEVICE))
                vl += nn.functional.mse_loss(pred, g['y'].to(DEVICE)).item()
                del pred
                torch.cuda.empty_cache()
            vl /= len(val_cases)
        val_losses.append(vl)
        elapsed = (time.time() - t0) / 60
        print(f'Ep {epoch+1:3d}/{CFG["epochs"]}  '
              f'train={train_losses[-1]:.4f}  val={vl:.4f}  '
              f'lr={scheduler.get_last_lr()[0]:.2e}  [{elapsed:.1f}min]')

    if (epoch + 1) % CFG['save_every'] == 0:
        torch.save({
            'epoch': epoch, 'model': model.state_dict(),
            'train_losses': train_losses, 'val_losses': val_losses,
        }, CKPT_DIR / f'species_epoch{epoch+1:04d}.pt')
        print(f'  → Checkpoint saved (epoch {epoch+1})')

print(f'\nTraining complete in {(time.time()-t0)/60:.1f} min')

## 7. Loss curves + TMA prediction sample

TMA = 0 at the wafer by boundary condition, so we visualise the **standoff gap**
(5 mm < z < 15 mm) where the precursor gradient is most informative.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(train_losses, label='train')
axes[0].plot(
    [i * CFG['log_every'] for i in range(len(val_losses))],
    val_losses, 'o--', label='val'
)
axes[0].set_yscale('log')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE (normalised TMA)')
axes[0].set_title('Species Head — Loss Curves')
axes[0].legend()

# TMA profile in standoff gap on one val case
model.eval()
case = val_cases[0]
g = make_graph(case)
with torch.no_grad():
    pred_norm = model(g['x'].to(DEVICE), g['edge_index'].to(DEVICE),
                     g['edge_attr'].to(DEVICE)).cpu().numpy().flatten()

TMA_pred = pred_norm * TMA_std + TMA_mean
TMA_true = case['TMA']
z        = case['coords'][:, 2]

# Standoff gap: 5mm < z < 15mm  (wafer at z=0, faceplate starts ~20mm)
mask = (z > 0.005) & (z < 0.015)
r    = np.sqrt(case['coords'][mask, 0]**2 + case['coords'][mask, 1]**2)
sort = np.argsort(r)

axes[1].scatter(r[sort]*1e3, TMA_true[mask][sort], s=1, alpha=0.3, label='CFD')
axes[1].scatter(r[sort]*1e3, TMA_pred[mask][sort], s=1, alpha=0.3, label='Pred')
axes[1].set_xlabel('r [mm]'); axes[1].set_ylabel('TMA mass fraction')
axes[1].set_title(f'TMA at standoff mid-plane — {case["name"]}')
axes[1].legend(markerscale=5)

plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'species_loss_curves.png'), dpi=120, bbox_inches='tight')
plt.show()

# Also print uniformity index of predicted TMA at wafer plane
wafer_mask = z < 0.002
if wafer_mask.sum() > 0:
    tma_w = TMA_pred[wafer_mask]
    if tma_w.mean() > 1e-8:
        ui = 1 - tma_w.std() / tma_w.mean()
    else:
        ui = float('nan')
    print(f'Predicted TMA uniformity at wafer: {ui:.4f}  (CFD BC = 0, so NaN is expected)')

print(f'TMA pred range: {TMA_pred.min():.4e} – {TMA_pred.max():.4e}')
print(f'TMA true range: {TMA_true.min():.4e} – {TMA_true.max():.4e}')

## 8. Save final checkpoint

In [ ]:
final_path = CKPT_DIR / 'species_head_final.pt'
torch.save({
    'model':            model.state_dict(),
    'cfg':              CFG,
    'norm': {
        'node_mean': node_mean.tolist(), 'node_std': node_std.tolist(),
        'TMA_mean':  TMA_mean,           'TMA_std':  TMA_std,
    },
    'final_train_loss': train_losses[-1],
    'final_val_loss':   val_losses[-1],
    'n_cases':          len(all_cases),
    'data_source':      'reactingFoam_openfoam_80cases',
}, final_path)

print(f'Saved: {final_path}')
print(f'Final train MSE: {train_losses[-1]:.4f}')
print(f'Final val MSE:   {val_losses[-1]:.4f}')
print('\nReady for 05_multihead_merge.ipynb')